<img src="assets/img/chm-banner.jpg">

# Detecting deforestation and forest growth with RasterFlow

This notebook will guide you through detecting deforestation and forest growth by comparing canopy height estimates across time, using Wherobots RasterFlow and the [Meta and World Resources Institute's Canopy Height prediction model (Meta CHM v1)](https://github.com/facebookresearch/HighResCanopyHeight/)<sup>1</sup>. This is an open source regression model that can predict the height of the tree canopy from high resolution imagery. The result of the model is a raster where each pixel is the estimated tree canopy height in meters.

By running this model on imagery from two different time periods and computing the difference, we can identify areas of significant canopy loss (deforestation) and canopy growth. We can then extract deforestation pockets as vector polygons for further analysis.

## Running inference on high resolution imagery

The Meta CHM v1 model was trained on 50cm resolution data, so we will select imagery with a similar resolution for our testing.

The National Agriculture Imagery Program (NAIP) provides aerial imagery for the United States, capturing high-resolution images during the agricultural growing seasons. The NAIP dataset contains a mixture of resolutions (30cm, 60cm and 1m).  See [this map](https://esri.maps.arcgis.com/apps/mapviewer/index.html?webmap=6cc0dcb225de4cb8aaa23c6a9cb59db8) for more details.  We will use 60cm imagery to test this model.

## Selecting an Area of Interest (AOI)
We will choose an Area of Interest (AOI) for our analysis. 

The area around Nashua, NH has a combination of urban settings, parks and forests so we will try out the model there. From the map, we see that 60cm imagery for New Hampshire was captured in both 2018 and 2021, so we will use these two time periods to detect change.

In [3]:
import wkls
import geopandas as gpd
import os

# Generate a geometry for Nashua, NH using WKLS (https://github.com/wherobots/wkls)
gdf = gpd.read_file(wkls.us.nh.nashua.geojson())

# Save the geometry to a parquet file in the user's S3 path
aoi_path = os.getenv("USER_S3_PATH") + "nashua.parquet"
gdf.to_parquet(aoi_path)

# we'll make variables for the bounds of our aoi for use after running inference to visualize results
min_lon, min_lat, max_lon, max_lat = gdf.total_bounds

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Initializing the RasterFlow client

In [2]:
from datetime import datetime

from rasterflow_remote import RasterflowClient

from rasterflow_remote.data_models import (
    ModelRecipes
)

rf_client = RasterflowClient()

## Running the model on two time periods

RasterFlow has pre-defined workflows to simplify orchestration of the processing steps for model inference.  These steps include:
* Ingesting imagery for the specified Area of Interest (AOI)
* Generating a seamless image from multiple image tiles (a mosaic) 
* Running inference with the selected model

The output is a Zarr store of the model outputs.

We will run the CHM model twice — once for 2021 imagery and once for 2018 imagery — so we can compare canopy height between the two periods.

Note: Each model run will take approximately 20 minutes to complete.

In [3]:
model_outputs_2021 = rf_client.build_and_predict_mosaic_recipe(
    # Path to our AOI in GeoParquet or GeoJSON format
    aoi = aoi_path,

    # Date range for imagery to be used by the model
    start = datetime(2021, 1, 1),
    end = datetime(2022, 1, 1),

    # Coordinate Reference System EPSG code for the output mosaic   
    crs_epsg = 3857,

    # The model recipe to be used for inference (CHM in this case)
    model_recipe = ModelRecipes.META_CHM_V1
)

print(model_outputs_2021)

🌎 [8s] SUCCEEDED | ID: ad6m6kdmmnp75vhztl7n
s3://wbts-temp-user-data-aws-us-west-2-prod/dfqlwcrxlk/rasterflow/rasterflow/production/fx1r1lfjam6wzi/3b59aa9a61dc.zarr


In [4]:
model_outputs_2018 = rf_client.build_and_predict_mosaic_recipe(
    # Path to our AOI in GeoParquet or GeoJSON format
    aoi = aoi_path,

    # Date range for imagery to be used by the model
    start = datetime(2018, 1, 1),
    end = datetime(2019, 1, 1),

    # Coordinate Reference System EPSG code for the output mosaic   
    crs_epsg = 3857,

    # The model recipe to be used for inference (CHM in this case)
    model_recipe = ModelRecipes.META_CHM_V1
)

print(model_outputs_2018)

🌎 [8s] SUCCEEDED | ID: ap5nwqgfcqr5tfgbqjs6
s3://wbts-temp-user-data-aws-us-west-2-prod/dfqlwcrxlk/rasterflow/rasterflow/production/fprl5j2g5smrw1/2e8b161dbdc0.zarr


## Visualize the canopy height for each time period
We will use hvplot and datashader to visualize the canopy height model outputs for 2021 and 2018. This lets us inspect the raw model predictions before computing the change between the two periods.

In [1]:
model_outputs_2018 = "s3://wbts-temp-user-data-aws-us-west-2-prod/dfqlwcrxlk/rasterflow/rasterflow/production/fprl5j2g5smrw1/2e8b161dbdc0.zarr"
model_outputs_2021 = "s3://wbts-temp-user-data-aws-us-west-2-prod/dfqlwcrxlk/rasterflow/rasterflow/production/fx1r1lfjam6wzi/3b59aa9a61dc.zarr"

In [ ]:
# Import libraries for visualization and coordinate transformation
import hvplot.xarray
import xarray as xr
import s3fs 
import zarr
from pyproj import Transformer
from holoviews.element.tiles import EsriImagery 

# Open the Zarr store
fs = s3fs.S3FileSystem(profile="default", asynchronous=True)
zstore = zarr.storage.FsspecStore(fs, path=model_outputs_2021[5:])
ds = xr.open_zarr(zstore)

# Create a transformer to convert from lat/lon to meters
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# Transform bounding box coordinates from lat/lon to meters
(min_x, max_x), (min_y, max_y) = transformer.transform(
    [min_lon, max_lon], 
    [min_lat, max_lat]
)

# Select the height variable and slice the dataset to the bounding box
# y=slice(max_y, min_y) handles the standard "North-to-South" image orientation
ds_subset = ds.sel(band="height",
    x=slice(min_x, max_x), 
    y=slice(max_y, min_y) 
)

# Select the first time step and extract the variables array
arr_subset_2021 = ds_subset.isel(time=0)["variables"].compute()

# Create a base map layer using Esri satellite imagery
base_map = EsriImagery()

# Create an overlay layer from the model outputs with hvplot
output_layer = arr_subset_2021.hvplot(
    x = "x",
    y = "y",
    geo = True,           # Enable geographic plotting
    dynamic = True,       # Enable dynamic rendering for interactivity
    rasterize = True,     # Use datashader for efficient rendering of large datasets
    cmap = "viridis",     # Color map for visualization
    aspect = "equal",     # Maintain equal aspect ratio
    title = "Canopy Height 2021" 
).opts(
    width = 600, 
    height = 600,
    alpha = 0.7           # Set transparency to see the base map underneath
)

# Combine the base map and output layer
final_plot = base_map * output_layer
final_plot

In [ ]:
# Open the 2018 Zarr store
zstore_2018 = zarr.storage.FsspecStore(fs, path=model_outputs_2018[5:])
ds_2018 = xr.open_zarr(zstore_2018)

# Select the height variable and slice the dataset to the bounding box
ds_subset_2018 = ds_2018.sel(band="height",
    x=slice(min_x, max_x), 
    y=slice(max_y, min_y) 
)

# Select the first time step and extract the variables array
arr_subset_2018 = ds_subset_2018.isel(time=0)["variables"].compute()

# Create a base map layer using Esri satellite imagery
base_map = EsriImagery()

# Create an overlay layer from the model outputs with hvplot
output_layer = arr_subset_2018.hvplot(
    x = "x",
    y = "y",
    geo = True,           # Enable geographic plotting
    dynamic = True,       # Enable dynamic rendering for interactivity
    rasterize = True,     # Use datashader for efficient rendering of large datasets
    cmap = "viridis",     # Color map for visualization
    aspect = "equal",     # Maintain equal aspect ratio
    title = "Canopy Height 2018" 
).opts(
    width = 600, 
    height = 600,
    alpha = 0.7           # Set transparency to see the base map underneath
)

# Combine the base map and output layer
final_plot = base_map * output_layer
final_plot

## Computing canopy height change

Now we subtract the 2018 canopy height from the 2021 canopy height. The resulting raster contains per-pixel height change in meters:
* **Negative values** indicate canopy loss (deforestation, clearing, or die-off)
* **Positive values** indicate canopy growth

We will visualize each direction separately to highlight areas of forest loss and forest growth.

In [8]:
canopy_change = (arr_subset_2021 - arr_subset_2018).compute()
canopy_change.name = "height_change_m"

In [ ]:
# Visualize canopy height loss (deforestation): clip positive values to zero
base_map_change = EsriImagery()

change_layer = canopy_change.clip(None, 0).hvplot(
        x = "x",
        y = "y",
        geo = True,           # Enable geographic plotting
        dynamic = True,       # Enable dynamic rendering for interactivity
        rasterize = True,     # Use datashader for efficient rendering of large datasets
        cmap = "hot_r",       # High-contrast heatmap: cool=no change, hot=large loss
        aspect = "equal",     # Maintain equal aspect ratio
        title = "Canopy Height Loss 2018 → 2021 (meters)"
).opts(
        width = 600,
        height = 600,
        alpha = 0.75,         # Set transparency to see the basemap underneath
        colorbar = True,
        clabel = "Height Change (m)"
)

# Combine the base map and change layer
change_plot = base_map_change * change_layer
change_plot

In [ ]:
# Visualize canopy height growth: clip negative values to zero
base_map_growth = EsriImagery()

growth_layer = canopy_change.clip(0, None).hvplot(
        x = "x",
        y = "y",
        geo = True,           # Enable geographic plotting
        dynamic = True,       # Enable dynamic rendering for interactivity
        rasterize = True,     # Use datashader for efficient rendering of large datasets
        cmap = "Greens",      # Green color map for forest growth
        aspect = "equal",     # Maintain equal aspect ratio
        title = "Canopy Height Growth 2018 → 2021 (meters)"
).opts(
        width = 600,
        height = 600,
        alpha = 0.75,         # Set transparency to see the basemap underneath
        colorbar = True,
        clabel = "Height Change (m)"
)

# Combine the base map and growth layer
growth_plot = base_map_growth * growth_layer
growth_plot

## Extracting deforestation pockets as polygons

To make the deforestation results actionable, we can convert areas of significant canopy loss into vector polygons. We apply a threshold (e.g., −20 meters) to identify pixels with substantial height loss, then use `rasterio.features.shapes` to vectorize the resulting binary mask into polygon geometries.

The resulting GeoDataFrame contains individual deforestation pockets that can be used for further spatial analysis, reporting, or integration with other geospatial workflows.

In [ ]:
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import shape

# Threshold in meters: pixels with canopy loss exceeding this value are flagged
threshold = -20

# Create a binary deforestation mask
mask = (canopy_change < threshold).astype(np.uint8)

# Convert raster mask to vector polygons
polygons = list(rasterio.features.shapes(mask.values, transform=mask.rio.transform(), connectivity=8))
gdf = gpd.GeoDataFrame(
    [{"geometry": shape(geom), "value": int(val)} for geom, val in polygons],
    crs=mask.rio.crs
).query("value == 1").reset_index(drop=True)

# Dissolve adjacent polygons into contiguous deforestation pockets
gdf = (
    gdf
    .dissolve()                          # merge all into one MultiPolygon
    .explode(index_parts=False)          # split back into individual groups
    .reset_index(drop=True)
)

print(f"Deforestation pockets detected: {len(gdf)}")
gdf.head()

## Visualize deforestation pockets

Finally, we overlay the extracted deforestation polygons on a satellite base map to see where significant canopy loss occurred between 2018 and 2021.

In [ ]:
import hvplot.pandas

# Reproject to EPSG:4326 for tile alignment
base_map_deforestation = EsriImagery()

deforestation_layer = gdf.to_crs("EPSG:4326").hvplot(
    geo=True,
    color="red",
    alpha=0.6,
    line_color="orangered",
    line_width=0.5,
    width=700,
    height=700,
    title="Deforestation Pockets 2018 → 2021 (change < −20 m)",
    legend=False
)

# Combine base map and deforestation layer
deforestation_plot = base_map_deforestation * deforestation_layer
deforestation_plot

### References

1. **Tolan, J., Yang, H.-I., Nosarzewski, B., Couairon, G., Vo, H. V., Brandt, J., Spore, J., Majumdar, S., Haziza, D., Vamaraju, J., et al. (2024).** Very high resolution canopy height maps from RGB imagery using self-supervised vision transformer and convolutional decoder trained on aerial lidar. *Remote Sensing of Environment*, 300, 113888.